# Section 04: 机器翻译（Seq2Seq）核心总结

## 任务定义
**翻译** = Seq2Seq（序列到序列）任务，输入一种语言的句子，输出另一种语言。

## 本节任务
用 `Helsinki-NLP/opus-mt-en-fr`（基于 MarianMT 的 en→fr 翻译模型）在 **KDE4** 数据集上微调。

## Seq2Seq 架构
```
                    ┌──────────────────────────────────┐
英文句子             │  Encoder (编码源语言)              │
"Hello world"  →   │  [CLS] Hello world [EOS]  →  记忆  │
                    └──────────────────────────────────┘
                                    ↓ 交叉注意力
                    ┌──────────────────────────────────┐
法文句子             │  Decoder (自回归生成目标语言)       │
"Bonjour monde"←   │  [BOS] Bonjour monde [EOS]        │
                    └──────────────────────────────────┘
```

## 完整流程
```
KDE4 数据集（英文+法文对）
    ↓ preprocess_function（源语言+目标语言分别 tokenize）
    ↓ DataCollatorForSeq2Seq（自动构造 decoder_input_ids）
    ↓ AutoModelForSeq2SeqLM
    ↓ Seq2SeqTrainer（predict_with_generate=True）
    ↓ 评估指标：BLEU
```

---
## 第一步：数据集准备

In [ ]:
from datasets import load_dataset

# KDE4 是软件本地化数据集，包含英文-法文平行句子对
raw_datasets = load_dataset("kde4", lang1="en", lang2="fr")

# 只有 train split，手动切分 validation
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)
split_datasets["validation"] = split_datasets.pop("test")  # 重命名

# 每个样本是一个翻译对
print(split_datasets["train"][1]["translation"])
# {'en': 'Default to expanded threads', 'fr': 'Par défaut, développer les fils de discussion'}

In [ ]:
# 用已有模型快速验证翻译质量（baseline）
from transformers import pipeline

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
translator = pipeline("translation", model=model_checkpoint)
print(translator("Default to expanded threads"))
# [{'translation_text': 'Par défaut pour les threads élargis'}]  ← 微调前效果一般

---
## 第二步（关键）：双语 Tokenize — as_target_tokenizer

翻译任务的 tokenize 比单任务复杂：源语言和目标语言需要**分别 tokenize**。

为什么？MarianMT 的 encoder/decoder 各自有不同的词表和特殊处理逻辑，
用 `as_target_tokenizer()` 上下文管理器切换到 decoder 模式。

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

en_sentence = "Default to expanded threads"
fr_sentence = "Par défaut, développer les fils de discussion"

# 用源语言模式（encoder）tokenize 英文
inputs = tokenizer(en_sentence)

# 用目标语言模式（decoder）tokenize 法文
with tokenizer.as_target_tokenizer():
    targets = tokenizer(fr_sentence)

# 对比：不用 as_target_tokenizer 的结果（错误）
wrong_targets = tokenizer(fr_sentence)
print("错误方式:", tokenizer.convert_ids_to_tokens(wrong_targets["input_ids"]))
print("正确方式:", tokenizer.convert_ids_to_tokens(targets["input_ids"]))
# 可以看到：正确方式的 tokenization 更准确（词汇边界处理不同）

In [ ]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    """
    批量处理翻译对：
    - 源语言（英文）→ input_ids
    - 目标语言（法文）→ labels（供计算 loss）
    """
    inputs  = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]

    # encoder 输入
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # decoder 目标（注意 as_target_tokenizer）
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = split_datasets.map(
    preprocess_function, batched=True,
    remove_columns=split_datasets["train"].column_names,
)

---
## 第三步：DataCollatorForSeq2Seq — 自动构造 decoder_input_ids

Seq2Seq 训练时，decoder 的输入是目标序列**右移一位**（Teacher Forcing）：
```
labels:           [A, B, C, EOS]   ← 训练目标（模型要预测的）
decoder_input_ids:[BOS, A, B, C]   ← decoder 的实际输入（每步用上一个真实词）
```
`DataCollatorForSeq2Seq` 自动完成这个右移操作，同时对变长序列做 padding。

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# model 参数用于让 collator 知道 pad_token_id（decoder padding 用）
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 查看 batch 的 key：多了 decoder_input_ids！
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
print("Batch keys:", list(batch.keys()))
print("labels:", batch["labels"])
print("decoder_input_ids:", batch["decoder_input_ids"])
# labels 中用 -100 padding（不计 loss），decoder_input_ids 用 pad_token_id

---
## 第四步：评估指标 — BLEU

**BLEU (Bilingual Evaluation Understudy)**：翻译任务的标准评估指标
- 计算预测译文与参考译文之间的 n-gram 重叠度
- 范围 0~100，越高越好
- 考虑 1-gram、2-gram、3-gram、4-gram 的精确率，再用简洁惩罚因子调整

**BLEU 的局限性**：
- 只看词语重叠，不考虑语义
- 对翻译顺序敏感
- 多个参考译文时更准确

In [ ]:
from datasets import load_metric
import numpy as np

metric = load_metric("sacrebleu")  # sacrebleu 是 BLEU 的标准实现

# 演示 BLEU 计算
predictions = ["This plugin lets you translate web pages between several languages automatically."]
references  = [["This plugin allows you to automatically translate web pages between several languages."]]
print(metric.compute(predictions=predictions, references=references))
# score 约 46.75

In [ ]:
def compute_metrics(eval_preds):
    """
    Seq2Seq 模型输出是 token id 序列，需要 decode 成文本再计算 BLEU。
    注意：labels 中 -100 需要替换为 pad_token_id 才能 decode。
    """
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # -100 无法 decode，先替换为 pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]  # sacrebleu 要求列表的列表

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

---
## 第五步：使用 Seq2SeqTrainer 训练

`Seq2SeqTrainer` 继承自 `Trainer`，关键额外参数：
- `predict_with_generate=True`：评估时用 `model.generate()` 生成序列（而不是取 argmax），更准确

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

args = Seq2SeqTrainingArguments(
    output_dir="marian-finetuned-kde4-en-to-fr",
    evaluation_strategy="no",          # 只在最后评估（翻译评估耗时）
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,        # 关键：用 generate() 而不是 argmax
    fp16=True,
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 训练前 BLEU: 39.27 → 训练后 BLEU: 52.94
# trainer.evaluate(max_length=max_target_length)
# trainer.train()

---
## 第六步：Accelerate 自定义循环的关键差异

Seq2Seq 的 Accelerate 循环与 Token Classification 类似，但评估时有重要区别：
不能用 `outputs.logits.argmax()`，必须用 `model.generate()` 生成序列。

In [ ]:
# Accelerate 评估阶段的关键代码（训练阶段与 section-02 完全相同）
import torch

# model.eval()
# for batch in eval_dataloader:
#     with torch.no_grad():
#         # ⚠️ 关键：用 generate() 而不是 forward()
#         # 必须先 unwrap_model，因为 generate() 不被 DDP 包装支持
#         generated_tokens = accelerator.unwrap_model(model).generate(
#             batch["input_ids"],
#             attention_mask=batch["attention_mask"],
#             max_length=128,
#         )
#
#     # 多 GPU 时，不同 batch 生成序列长度不一，需要 pad 后 gather
#     generated_tokens = accelerator.pad_across_processes(
#         generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
#     )
#     labels = accelerator.pad_across_processes(batch["labels"], dim=1, pad_index=-100)
#
#     predictions_gathered = accelerator.gather(generated_tokens)
#     labels_gathered = accelerator.gather(labels)

print("核心差异：")
print("  Token Classification: outputs.logits.argmax()")
print("  Seq2Seq Translation:  accelerator.unwrap_model(model).generate(...)")

---
## 总结

### 核心知识点速查

| 概念 | 说明 |
|------|------|
| Seq2Seq 架构 | Encoder 读取源语言 → Decoder 生成目标语言，通过交叉注意力连接 |
| `as_target_tokenizer()` | 切换 tokenizer 到 decoder 模式，目标语言 tokenize 必须使用 |
| Teacher Forcing | 训练时 decoder 输入是真实目标序列（右移），而非上一步预测结果 |
| `DataCollatorForSeq2Seq` | 自动构造 `decoder_input_ids`（labels 右移+BOS），并 pad |
| `predict_with_generate=True` | 评估用 autoregressive generate()，比 argmax 更准确 |
| BLEU 分数 | 越高越好；微调前 39.27 → 微调后 52.94（+13 BLEU）|

### 与 Token Classification 的对比
```
Token Classification：
  - 模型：AutoModelForTokenClassification
  - 输出：每个 token 的分类 logit
  - 评估：logits.argmax()

Seq2Seq Translation：
  - 模型：AutoModelForSeq2SeqLM
  - 输出：自回归生成的 token 序列
  - 评估：model.generate()（beam search 等策略）
```